In [19]:
import librosa
import torch
import os
from pyannote.audio import Pipeline
import soundfile as sf
import whisper
import json
from openai import OpenAI
import re

In [2]:
# GPU 사용 가능 여부 확인 및 장치 설정
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU 사용 가능. 장치 이름: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU 사용 불가능. CPU 사용.")

# Whisper 모델 로드 및 장치 이동
try:
    model = whisper.load_model("base")
    model = model.to(device)
    print(f"Whisper 모델 로드 및 {device} 이동 완료.")
except Exception as e:
    print(f"Whisper 모델 로드 실패: {e}")

GPU 사용 가능. 장치 이름: NVIDIA GeForce RTX 4060 Ti
Whisper 모델 로드 및 cuda 이동 완료.


In [3]:
# pyannote-segmentation 파이프라인 초기화 및 GPU 설정
try:
    pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.0", use_auth_token="YOUR_HuggingFace_API_KEY") # 필요시 Hugging Face 토큰 사용
    pipeline.to(torch.device(device))
    print("pyannote-segmentation 파이프라인 로드 및 GPU 이동 완료.")
except Exception as e:
    print(f"pyannote-segmentation 파이프라인 로드 실패: {e}")

INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
C:\Users\tlfqj\anaconda3\envs\cuda\lib\inspect.py:869: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):


pyannote-segmentation 파이프라인 로드 및 GPU 이동 완료.


C:\Users\tlfqj\anaconda3\envs\cuda\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [4]:
# Whisper 모델 로드 및 GPU 이동
try:
    model = whisper.load_model("base").to(device) # 원하는 모델 크기 선택 (tiny, base, small, medium, large)
    print("Whisper 모델 로드 및 GPU 이동 완료.")
except Exception as e:
    print(f"Whisper 모델 로드 실패: {e}")

Whisper 모델 로드 및 GPU 이동 완료.


In [5]:
# 오디오 파일 경로 설정
audio_path = "C:/Users/tlfqj/OneDriveDesktop/한신대 프로젝트/AI BAD WORD/이문식의-쩌는-찰진욕-_라이타를-켜라_-gil-bong.mp3"

In [6]:
try:
    audio, sample_rate = sf.read(audio_path)
    print(f"오디오 파일 '{audio_path}' 로드 완료 (샘플링 속도: {sample_rate} Hz)")
except Exception as e:
    print(f"오디오 파일 로드 실패: {e}")
    exit()

오디오 파일 'C:/Users/tlfqj/OneDriveDesktop/한신대 프로젝트/AI BAD WORD/이문식의-쩌는-찰진욕-_라이타를-켜라_-gil-bong.mp3' 로드 완료 (샘플링 속도: 48000 Hz)


In [13]:
segments = []
segment_duration = 10  # 음성 인식 처리 단위 (초)

try:
    diarization = pipeline(audio_path)

    print("\n화자 분리 결과:")
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        print(f"[{segment.start:.2f} - {segment.end:.2f}] {speaker}")

    print("\n화자별 음성 인식 결과 (작은 세그먼트):")
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        start = segment.start
        end = segment.end
        num_sub_segments = int((end - start) / segment_duration) + 1
        for i in range(num_sub_segments):
            sub_start = start + i * segment_duration
            sub_end = min(end, sub_start + segment_duration)

            sub_start_time = int(sub_start * sample_rate)
            sub_end_time = int(sub_end * sample_rate)
            speaker_sub_segment = audio[sub_start_time:sub_end_time]

            if sample_rate != 16000:
                speaker_sub_segment = librosa.resample(speaker_sub_segment, orig_sr=sample_rate, target_sr=16000)

            speaker_sub_segment_tensor = torch.from_numpy(speaker_sub_segment).float().to(device)
            result = model.transcribe(speaker_sub_segment_tensor, language='ko')

            segments.append({
                'start': sub_start,
                'end': sub_end,
                'speaker': speaker,
                'text': result['text']
            })

    print("\n최종 음성 인식 결과 (작은 세그먼트 병합):")
    output_text = ""
    for seg in segments:
        output_text += f"[{seg['start']:.2f} - {seg['end']:.2f}] {seg['speaker']}: {seg['text']}\n"
        print(f"[{seg['start']:.2f} - {seg['end']:.2f}] {seg['speaker']}: {seg['text']}")


except Exception as e:
    print(f"처리 실패: {e}")




화자 분리 결과:
[0.03 - 1.53] SPEAKER_01
[2.24 - 3.41] SPEAKER_01
[4.05 - 7.39] SPEAKER_01
[8.69 - 12.35] SPEAKER_01
[12.96 - 14.29] SPEAKER_01
[14.95 - 19.76] SPEAKER_01
[20.08 - 20.10] SPEAKER_00
[20.10 - 20.74] SPEAKER_01
[21.23 - 21.24] SPEAKER_01
[21.24 - 21.31] SPEAKER_00
[21.31 - 21.75] SPEAKER_01
[21.75 - 22.54] SPEAKER_00
[23.27 - 25.12] SPEAKER_00
[23.88 - 24.94] SPEAKER_01
[25.60 - 29.29] SPEAKER_00
[26.66 - 26.95] SPEAKER_01

화자별 음성 인식 결과 (작은 세그먼트):

최종 음성 인식 결과 (작은 세그먼트 병합):
[0.03 - 1.53] SPEAKER_01:  돌아이 새끼를 봤나 이 ㅅ..
[2.24 - 3.41] SPEAKER_01:  야 이 존발한 새끼야
[4.05 - 7.39] SPEAKER_01:  이분이 어떤 분이신지 알고 개소를 하고 잡혀있어 이 시퍼르러 새끼야
[8.69 - 12.35] SPEAKER_01:  우리 형님이 저런 똥깔에서 바닥에 떨어진 싸우그레 라이트라 죽고 다니는 그런 분으로 보여?
[12.96 - 14.29] SPEAKER_01:  우리 언니 sixty 개의 microwave
[14.95 - 19.76] SPEAKER_01:  가뜩이나 요즘 가을 주고서 민감하신데 시발자로 새깨분이 파악하면 시부를 전만 안 닦다고 해야 할 것 같아요
[20.08 - 20.10] SPEAKER_00: 
[20.10 - 20.74] SPEAKER_01:  전역자들 2
[21.23 - 21.24] SPEAKER_01: 
[21.24 - 21.31] SPEAKER_00: 
[21.31 - 21.75] SPE

In [14]:
print(output_text)

[0.03 - 1.53] SPEAKER_01:  돌아이 새끼를 봤나 이 ㅅ..
[2.24 - 3.41] SPEAKER_01:  야 이 존발한 새끼야
[4.05 - 7.39] SPEAKER_01:  이분이 어떤 분이신지 알고 개소를 하고 잡혀있어 이 시퍼르러 새끼야
[8.69 - 12.35] SPEAKER_01:  우리 형님이 저런 똥깔에서 바닥에 떨어진 싸우그레 라이트라 죽고 다니는 그런 분으로 보여?
[12.96 - 14.29] SPEAKER_01:  우리 언니 sixty 개의 microwave
[14.95 - 19.76] SPEAKER_01:  가뜩이나 요즘 가을 주고서 민감하신데 시발자로 새깨분이 파악하면 시부를 전만 안 닦다고 해야 할 것 같아요
[20.08 - 20.10] SPEAKER_00: 
[20.10 - 20.74] SPEAKER_01:  전역자들 2
[21.23 - 21.24] SPEAKER_01: 
[21.24 - 21.31] SPEAKER_00: 
[21.31 - 21.75] SPEAKER_01:  경사�opp dura
[21.75 - 22.54] SPEAKER_00:  conscious
[23.27 - 25.12] SPEAKER_00:  쟤는 무슨if
[23.88 - 24.94] SPEAKER_01:  본mud지 flank Sur
[25.60 - 29.29] SPEAKER_00:  야! 안 돌아! 좀 저 딸시기 신경 쓸 때가 아니다! 가자!
[26.66 - 26.95] SPEAKER_01: 



# 수동 탐지

In [21]:
#수동 탐지
def count_profanity_by_speaker_from_text(text, profanity_list):
    """
    최종 음성 인식 결과 텍스트에서 화자별로 특정 욕설 단어들의 빈도를 세는 함수입니다.

    Args:
        text (str): 최종 음성 인식 결과 텍스트 (각 줄은 "[시작시간 - 종료시간] 화자: 발언" 형식).
        profanity_list (list): 욕설 단어 목록입니다.

    Returns:
        dict: 화자별 욕설 빈도를 담은 딕셔너리 ({"화자": {"단어": 빈도}})입니다.
    """
    speaker_profanity_counts = {}
    lines = text.strip().split('\n')

    for line in lines:
        match = re.match(r"^\[.+?\]\s(SPEAKER_\d+):\s(.+)$", line)
        if match:
            speaker = match.group(1)
            utterance = match.group(2).lower()

            if speaker not in speaker_profanity_counts:
                speaker_profanity_counts[speaker] = {}

            for word in profanity_list:
                pattern = r'\b' + re.escape(word.lower()) + r'\b'
                matches = re.findall(pattern, utterance)
                if matches:
                    if word not in speaker_profanity_counts[speaker]:
                        speaker_profanity_counts[speaker][word] = 0
                    speaker_profanity_counts[speaker][word] += len(matches)

    return speaker_profanity_counts

# 최종 음성 인식 결과 (이전 코드에서 output_text 변수에 저장된 내용)
print(output_text)

# 판별할 욕설 단어 목록 (원하는 단어를 추가/수정하세요)
korean_profanity = ["씨발", "개새끼", "좆", "병신", "새끼", "씹", "또라이", "존만", "이씨", "개소리", "새끼야"]

# 화자별 욕설 빈도 판별
speaker_profanity_frequency = count_profanity_by_speaker_from_text(output_text, korean_profanity)

# 결과 출력
print("화자별 욕설 단어 빈도:")
if speaker_profanity_frequency:
    for speaker, word_counts in speaker_profanity_frequency.items():
        print(f"\n{speaker}:")
        for word, count in word_counts.items():
            print(f"  '{word}': {count}번")
else:
    print("욕설이 감지되지 않았거나, 올바른 형식의 텍스트가 아닙니다.")

[0.03 - 1.53] SPEAKER_01:  돌아이 새끼를 봤나 이 ㅅ..
[2.24 - 3.41] SPEAKER_01:  야 이 존발한 새끼야
[4.05 - 7.39] SPEAKER_01:  이분이 어떤 분이신지 알고 개소를 하고 잡혀있어 이 시퍼르러 새끼야
[8.69 - 12.35] SPEAKER_01:  우리 형님이 저런 똥깔에서 바닥에 떨어진 싸우그레 라이트라 죽고 다니는 그런 분으로 보여?
[12.96 - 14.29] SPEAKER_01:  우리 언니 sixty 개의 microwave
[14.95 - 19.76] SPEAKER_01:  가뜩이나 요즘 가을 주고서 민감하신데 시발자로 새깨분이 파악하면 시부를 전만 안 닦다고 해야 할 것 같아요
[20.08 - 20.10] SPEAKER_00: 
[20.10 - 20.74] SPEAKER_01:  전역자들 2
[21.23 - 21.24] SPEAKER_01: 
[21.24 - 21.31] SPEAKER_00: 
[21.31 - 21.75] SPEAKER_01:  경사�opp dura
[21.75 - 22.54] SPEAKER_00:  conscious
[23.27 - 25.12] SPEAKER_00:  쟤는 무슨if
[23.88 - 24.94] SPEAKER_01:  본mud지 flank Sur
[25.60 - 29.29] SPEAKER_00:  야! 안 돌아! 좀 저 딸시기 신경 쓸 때가 아니다! 가자!
[26.66 - 26.95] SPEAKER_01: 

화자별 욕설 단어 빈도:

SPEAKER_01:
  '새끼야': 2번

SPEAKER_00:


# AI 탐지

In [39]:
#OpenAI API Key 설정
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [41]:
# OpenAI API로 욕설 탐지하는 함수

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def detect_profanity_with_openai(output_text):
    prompt = f"""
아래는 음성 인식 결과입니다.

각 줄은 다음 형식입니다.
[시작시간 - 종료시간] SPEAKER_ID: 발화 내용

너의 역할은 이 텍스트에서 욕설, 비속어, 모욕적 표현, 공격적인 표현을 탐지하는 것입니다.

반드시 아래 JSON 형식으로만 출력하세요.
설명 문장, 코드블록, ```json 은 절대 붙이지 마세요.

출력 형식:
{{
  "analysis_result": "전체 분석 결과 한 문장",
  "total": {{
    "detected": true,
    "total_count": 0,
    "summary": "전체 요약"
  }},
  "speakers": {{
    "SPEAKER_00": {{
      "count": 0,
      "items": [
        {{
          "time": "시작시간 - 종료시간",
          "expression": "탐지된 욕설 또는 비속어",
          "category": "욕설/비속어/모욕/의심 표현",
          "utterance": "해당 발화 전체 문장",
          "reason": "탐지 이유"
        }}
      ]
    }}
  }},
  "expressions": [
    {{
      "expression": "욕설 표현",
      "count": 0
    }}
  ]
}}

분석 기준:
1. 실제 텍스트에 등장한 표현만 탐지하세요.
2. 욕설, 비속어, 모욕적 표현, 공격적인 표현을 탐지하세요.
3. 화자와 시간 구간을 반드시 포함하세요.
4. 같은 표현이 여러 번 등장하면 횟수를 집계하세요.
5. 애매한 표현은 "의심 표현"으로 분류하세요.

분석할 텍스트:
\"\"\"
{output_text}
\"\"\"
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 욕설, 비속어, 공격적 표현을 분석하는 도구다. 반드시 JSON만 출력한다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()

        # 혹시 코드블록이 붙었을 경우 제거
        result_text = result_text.replace("```json", "").replace("```", "").strip()

        result = json.loads(result_text)
        return result

    except json.JSONDecodeError:
        print("JSON 변환 실패")
        print("모델 응답 원문:")
        print(result_text)
        return None

    except Exception as e:
        print(f"OpenAI 욕설 탐지 실패: {e}")
        return None

In [43]:
# 결과를 보기 좋게 출력하는 함수

def print_result_like_image(result):
    if result is None:
        print("분석 결과가 없습니다.")
        return

    print("분석 결과:")
    print(result.get("analysis_result", "분석 결과 없음"))
    print()

    total = result.get("total", {})
    print("=== 전체 욕설 탐지 결과 ===")
    print(f"- 욕설 탐지 여부: {total.get('detected', False)}")
    print(f"- 총 탐지 횟수: {total.get('total_count', 0)}회")
    print(f"- 요약: {total.get('summary', '요약 없음')}")
    print()

    speakers = result.get("speakers", {})
    print("=== 화자별 욕설 탐지 결과 ===")

    if not speakers:
        print("화자별 탐지 결과 없음")
    else:
        for speaker, data in speakers.items():
            print(f"\n[{speaker}]")
            print(f"- 탐지 횟수: {data.get('count', 0)}회")

            items = data.get("items", [])
            if not items:
                print("- 탐지된 표현 없음")
                continue

            for item in items:
                print(f"  - 시간: {item.get('time')}")
                print(f"    표현: {item.get('expression')}")
                print(f"    분류: {item.get('category')}")
                print(f"    이유: {item.get('reason')}")
    print()

    expressions = result.get("expressions", [])
    print("=== 욕설 단어/표현 목록 ===")

    if not expressions:
        print("탐지된 욕설 표현 없음")
    else:
        for item in expressions:
            print(f"- '{item.get('expression')}': {item.get('count', 0)}회")
    print()

    print("=== 욕설이 탐지된 원문 발화 구간 ===")

    has_utterance = False

    for speaker, data in speakers.items():
        for item in data.get("items", []):
            has_utterance = True
            print(f"[{item.get('time')}] {speaker}: {item.get('utterance')}")

    if not has_utterance:
        print("욕설이 포함된 발화 구간 없음")

In [45]:
# output_text가 이미 만들어져 있어야 함
print(output_text)

# OpenAI API로 욕설 탐지
result = detect_profanity_with_openai(output_text)

# 사진처럼 결과 출력
print_result_like_image(result)

[0.03 - 1.53] SPEAKER_01:  돌아이 새끼를 봤나 이 ㅅ..
[2.24 - 3.41] SPEAKER_01:  야 이 존발한 새끼야
[4.05 - 7.39] SPEAKER_01:  이분이 어떤 분이신지 알고 개소를 하고 잡혀있어 이 시퍼르러 새끼야
[8.69 - 12.35] SPEAKER_01:  우리 형님이 저런 똥깔에서 바닥에 떨어진 싸우그레 라이트라 죽고 다니는 그런 분으로 보여?
[12.96 - 14.29] SPEAKER_01:  우리 언니 sixty 개의 microwave
[14.95 - 19.76] SPEAKER_01:  가뜩이나 요즘 가을 주고서 민감하신데 시발자로 새깨분이 파악하면 시부를 전만 안 닦다고 해야 할 것 같아요
[20.08 - 20.10] SPEAKER_00: 
[20.10 - 20.74] SPEAKER_01:  전역자들 2
[21.23 - 21.24] SPEAKER_01: 
[21.24 - 21.31] SPEAKER_00: 
[21.31 - 21.75] SPEAKER_01:  경사�opp dura
[21.75 - 22.54] SPEAKER_00:  conscious
[23.27 - 25.12] SPEAKER_00:  쟤는 무슨if
[23.88 - 24.94] SPEAKER_01:  본mud지 flank Sur
[25.60 - 29.29] SPEAKER_00:  야! 안 돌아! 좀 저 딸시기 신경 쓸 때가 아니다! 가자!
[26.66 - 26.95] SPEAKER_01: 



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


분석 결과:
욕설과 비속어가 다수 탐지되었으며, 일부 표현은 모욕적이고 공격적인 성격을 띕니다.

=== 전체 욕설 탐지 결과 ===
- 욕설 탐지 여부: True
- 총 탐지 횟수: 6회
- 요약: 욕설과 비속어 6건이 SPEAKER_01의 발화에서 탐지됨

=== 화자별 욕설 탐지 결과 ===

[SPEAKER_01]
- 탐지 횟수: 6회
  - 시간: 0.03 - 1.53
    표현: 돌아이 새끼
    분류: 욕설/비속어
    이유: 돌아이, 새끼는 욕설 및 비속어로 분류됨
  - 시간: 2.24 - 3.41
    표현: 존발한 새끼야
    분류: 욕설/비속어
    이유: 존발한, 새끼야는 욕설 및 비속어임
  - 시간: 4.05 - 7.39
    표현: 개소를 하고 잡혀있어 이 시퍼르러 새끼야
    분류: 욕설/비속어
    이유: 개소, 시퍼르러, 새끼야는 욕설 및 비속어임
  - 시간: 8.69 - 12.35
    표현: 똥깔
    분류: 욕설/비속어
    이유: 똥깔은 비속어로 분류됨
  - 시간: 14.95 - 19.76
    표현: 시발자로 새깨분
    분류: 욕설/비속어
    이유: 시발자, 새깨분은 욕설 및 비속어임

=== 욕설 단어/표현 목록 ===
- '돌아이 새끼': 1회
- '존발한 새끼야': 1회
- '개소': 1회
- '시퍼르러 새끼야': 1회
- '똥깔': 1회
- '시발자로 새깨분': 1회

=== 욕설이 탐지된 원문 발화 구간 ===
[0.03 - 1.53] SPEAKER_01: 돌아이 새끼를 봤나 이 ㅅ..
[2.24 - 3.41] SPEAKER_01: 야 이 존발한 새끼야
[4.05 - 7.39] SPEAKER_01: 이분이 어떤 분이신지 알고 개소를 하고 잡혀있어 이 시퍼르러 새끼야
[8.69 - 12.35] SPEAKER_01: 우리 형님이 저런 똥깔에서 바닥에 떨어진 싸우그레 라이트라 죽고 다니는 그런 분으로 보여?
[14.95 - 19.76] SPEAKER_01: 가뜩이나 요즘 가을 주고